In [4]:
!pwd

/content


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd

# Read the SHAP csv file for predicted test data
## Retained only those test transactions where the predicted probability is >= 0.86

In [3]:
file_path = '/content/drive/MyDrive/Colab Notebooks/IEEE_CIS_Dataset/Output/Test_Feature_SHAP_Threshold_0.86.csv'
df_shap = pd.read_csv(file_path)

In [11]:
df_shap.shape

(11503, 94)

In [12]:
df_shap.head()

,TransactionID,Predicted_Fraud_Probability,PaymentCard_Dummy1,dist1,Count_C1,Count_C3,Count_C5,Days_Since_D1,Days_Since_D2,Days_Since_D4,...,Shap_GNN_Emb_6,Shap_GNN_Emb_7,Shap_GNN_Emb_8,Shap_GNN_Emb_9,Shap_GNN_Emb_10,Shap_GNN_Emb_11,Shap_GNN_Emb_12,Shap_GNN_Emb_13,Shap_GNN_Emb_14,Shap_GNN_Emb_15
0,3663602,0.903029,185,-999,8,0,0,-999,-999,316,...,-0.004650,-0.109538,-0.016028,0.445932,0.077392,-0.244019,0.176191,0.066261,0.339271,-0.126015
1,3663769,0.972774,185,-999,65,0,0,183,3,207,...,0.104075,0.079099,0.088738,-0.022575,-0.049313,-0.117689,-0.026835,0.079719,-0.012449,0.063552
2,3663773,0.962453,150,-999,3,0,0,0,-999,-999,...,0.020354,-0.060369,-0.069775,0.205876,-0.035330,-0.020629,0.135172,0.071598,-0.017270,-0.070597
3,3663778,0.958745,150,-999,4,0,0,0,-999,-999,...,0.015917,-0.095892,-0.050506,0.128934,-0.254020,-0.091572,0.094330,0.036449,-0.148856,-0.038734
4,3663780,0.969665,150,-999,4,0,0,0,-999,-999,...,0.034424,-0.180550,-0.062317,0.206255,-0.195552,0.048113,0.170974,0.077929,-0.067970,-0.144624


# Read reference transaction from Validation set
## The validation set had both true and false positives for the trained model. The refrence file contains 100 transactions each of TP/ FP/ TN/ FN so that the agent can guide the human analyst if the pattern in reference file matches that of test transactions and help identify any False Positives or False Negatives

In [4]:
file_path = '/content/drive/MyDrive/Colab Notebooks/IEEE_CIS_Dataset/Output/df_reference_sampled.csv'
df_validation_refrence = pd.read_csv(file_path)

In [14]:
df_validation_refrence.shape

(400, 102)

In [15]:
df_validation_refrence.head()

,TransactionID,isFraud,ProductCD,PaymentCard_Bank_ID,PaymentCard_Billing_Country,PaymentCard_Dummy1,PaymentCard_Type,PaymentCard_Category,PaymentCard_Network,addr1,...,Shap_GNN_Emb_6,Shap_GNN_Emb_7,Shap_GNN_Emb_8,Shap_GNN_Emb_9,Shap_GNN_Emb_10,Shap_GNN_Emb_11,Shap_GNN_Emb_12,Shap_GNN_Emb_13,Shap_GNN_Emb_14,Shap_GNN_Emb_15
0,3007488,1,C,9026,545.0,185.0,visa,137.0,credit,130.0,...,0.157601,-0.043488,0.067945,-0.030803,0.059729,-0.087095,-0.055452,0.004247,0.027778,0.057200
1,3430992,1,C,9633,130.0,185.0,visa,138.0,debit,130.0,...,0.073697,-0.141643,0.033660,-0.018703,0.062841,-0.251278,0.087303,0.132030,-0.124435,0.015594
2,3020822,1,C,10369,176.0,185.0,mastercard,137.0,credit,130.0,...,0.185694,0.034402,0.145059,-0.040445,-0.002290,-0.014703,0.041042,-0.000954,-0.067421,-0.112558
3,3028324,1,C,16062,500.0,185.0,mastercard,137.0,credit,130.0,...,0.001762,-0.034233,0.048398,-0.037358,-0.157761,-0.436732,-0.002132,0.159068,0.084533,0.027284
4,3006064,1,C,2256,545.0,185.0,visa,226.0,credit,161.0,...,0.128271,-0.040342,0.051573,-0.035678,0.099675,-0.039780,-0.074025,-0.009531,-0.003887,0.072463


# Agentic AI Framework: SHAP Interpreter & Fraud Triage
This section establishes the framework for the two-agent system.

**Dependencies needed**: You will need an LLM library (like `openai` or `langchain`). For this framework, we define standard python functions that can wrap LLM calls.

In [10]:
# !pip install crewai langchain-openai python-dotenv

import os
from dotenv import load_dotenv

# Since you are running a Google Colab kernel, the code executes on a cloud VM, 
# not on your local Windows machine. 
# Therefore it cannot see the local `.env` file in your VS Code folder.

# SOLUTION: 
# 1. Create a file named .env in your Google Drive under "Colab Notebooks"
# 2. Add your OPENAI_API_KEY=sk-.... to it.
env_path = '/content/drive/MyDrive/Colab Notebooks/IEEE_CIS_Dataset/.env'

load_dotenv(env_path)

if not os.environ.get("OPENAI_API_KEY"):
    print(f"Warning: OPENAI_API_KEY not found! Please ensure you have added it to {env_path} in your Google Drive.")
else:
    print("API Key loaded successfully!")

API Key loaded successfully!


In [2]:
!pip install crewai langchain-openai faiss-cpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 4.4 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opentelemetry-exporter-otlp-proto-grpc to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.0/195.0 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.1/122.1 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 78.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━

In [12]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 36.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 34.3 MB/s eta 0:00:0000:0100:01


In [ ]:
import faiss
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from crewai import Agent, Task, Crew, Process

def build_vector_db(df_validation):
    """
    Builds a FAISS Vector Database for Nearest Neighbor search.
    Encodes all SHAP feature columns into a high-dimensional vector space.
    """
    # Isolate all SHAP columns and sort alphabetically for consistency
    shap_cols = [c for c in df_validation.columns if str(c).startswith('Shap_')]
    shap_cols.sort()
    
    # Extract the numeric values for the FAISS index
    # (FAISS requires float32 contiguous arrays)
    vectors = np.ascontiguousarray(df_validation[shap_cols].fillna(0).values.astype('float32'))
    
    # Build FAISS Index (L2 = Euclidean Distance)
    dim = vectors.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(vectors)
    
    # Keep track of the categories assigned to these vectors
    categories = df_validation['Category'].values if 'Category' in df_validation.columns else np.array(['Unknown'] * len(df_validation))
    
    return index, shap_cols, categories

async def run_crewai_fraud_analysis(transaction_id, df_shap, vector_db):
    """
    CrewAI Framework orchestrating the SHAP Interpreter and Fraud Triage agents.
    """
    index, shap_cols, categories = vector_db
    
    # 1. Fetch Transaction Data
    tx_data = df_shap[df_shap['TransactionID'] == transaction_id]
    if tx_data.empty: 
        return f"Transaction {transaction_id} not found."
    
    probability = tx_data['Predicted_Fraud_Probability'].values[0] if 'Predicted_Fraud_Probability' in tx_data.columns else "N/A"
    
    # For Interpret Agent: Get the top 5 highest contributing SHAP values
    top_5_series = tx_data[[c for c in shap_cols if c in tx_data.columns]].iloc[0].sort_values(ascending=False).head(5)
    tx_features_top_5 = top_5_series.to_dict()
    
    # --- Plot the Top 5 SHAP values as a horizontal bar graph ---
    plt.figure(figsize=(10, 5))
    top_5_series.sort_values(ascending=True).plot(kind='barh', color='salmon')
    plt.title(f'Top 5 SHAP Features for Transaction {transaction_id}')
    plt.xlabel('SHAP Contribution Value')
    plt.ylabel('Feature')
    plt.tight_layout()
    plt.show()
    # ------------------------------------------------------------
    
    # 2. Vector DB Approximate Nearest Neighbor Search
    # Align the test transaction completely to the Vector DB's expected columns using **ALL SHAP variables**
    tx_vector_df = tx_data.reindex(columns=shap_cols, fill_value=0)
    tx_vector = np.ascontiguousarray(tx_vector_df.values.astype('float32'))
    
    # Perform Search (Top 5 matches via Euclidean distance using the entire multi-dimensional SHAP space)
    distances, indices = index.search(tx_vector, 5)
    
    # Map matched indices back to their target categories
    closest_categories = categories[indices[0]]
    most_likely_category = pd.Series(closest_categories).mode()[0]
    
    reference_data_sample_str = f"Top 5 ANN Matching Categories from Vector DB (Based on ALL SHAP features): {closest_categories.tolist()}\nEuclidean Distances: {distances[0].tolist()}\n\n--> ANN Selected Category: {most_likely_category}"
    
    # ==========================================
    # Define Agents
    # ==========================================
    shap_interpreter = Agent(
        role='SHAP Interpreter Analyst',
        goal='Interpret SHAP values to explain the key factors driving high fraud probabilities.',
        backstory='You are an expert Data Scientist specializing in model explainability and fraud detection. You break down complex feature attributes into simple business risk factors.',
        verbose=True,
        allow_delegation=False,
        llm="gpt-4o-mini"
    )

    fraud_triage_specialist = Agent(
        role='Fraud Triage Specialist',
        goal='Provide an actionable recommendation based on SHAP explanations and Vector DB nearest-neighbor search.',
        backstory='You are a Senior Fraud Analyst. You evaluate explainability metrics and cross-reference them with historical patterns retrieved from Vector DBs to make final block/allow decisions.',
        verbose=True,
        allow_delegation=False,
        llm="gpt-4o-mini"
    )
    
    # ==========================================
    # Define Tasks
    # ==========================================
    interpret_task = Task(
        description=f'''
        Analyze Transaction {transaction_id} with Fraud Probability: {probability}.
        The top driving SHAP features and their values are: {tx_features_top_5}.
        
        Provide a concise summary explaining to a human analyst why this transaction was flagged.
        ''',
        expected_output='A brief paragraph explaining the main reasons the model predicted high fraud risk.',
        agent=shap_interpreter
    )

    triage_task = Task(
        description=f'''
        Review the SHAP interpretation for Transaction {transaction_id} (Probability: {probability}).
        Here is the historical pattern matching result from the FAISS Vector Database search using Euclidean distance on ALL SHAP vectors: 
        {reference_data_sample_str}

        Combine the SHAP context with the Vector DB match to decide the best action. 
        If the likely category is FP (False Positive), lean towards recommendations like "Request 3D authentication" or "Allow with monitoring". 
        If TP (True Positive), lean towards "Keep blocked".
        If FN (False Negative), lean towards "Manual Review. Even though the probability is low, it might be a fraud transaction".
        ''',
        expected_output=f'''
        Return the output STRICTLY in the following format:
        1. Recommendation: [e.g. Request 3D authentication / Manual review / keep blocked]
        2. Fraud probability: [{probability}]
        3. Justification: [Your reasoning factoring in the matching {most_likely_category} category...]
        ''',
        agent=fraud_triage_specialist
    )

    # ==========================================
    # Assemble & Run Crew
    # ==========================================
    crew = Crew(
        agents=[shap_interpreter, fraud_triage_specialist],
        tasks=[interpret_task, triage_task],
        verbose=True,
        process=Process.sequential
    )

    print(f"\n--- Initiating CrewAI Analysis for Transaction: {transaction_id} ---")
    result = await crew.kickoff_async()
    return result

In [11]:
# --- Example Usage --- 
# Ensure you have installed the required libraries and set the API Key first.

# 1. Build the Vector DB globally once
# (This loads all Shap_xx variables into a FAISS index and links them to the Category column)
vector_db = build_vector_db(df_validation_refrence)

# 2. Pick a transaction
sample_id = df_shap['TransactionID'].iloc[0] if 'TransactionID' in df_shap.columns else "TEST_ID_1"

# 3. Run the crew analysis asynchronously with the Vector DB
final_report = await run_crewai_fraud_analysis(sample_id, df_shap, vector_db)
print(final_report)


--- Initiating CrewAI Analysis for Transaction: 3663602 ---


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 1d7764a8-4d45-46cd-bb76-4912c612f8cd                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│          Analyze Transaction 3663602 with Fraud Probability: 0.903028677.                                       │
│          The top driving SHAP features and their values are: {'Shap_Count_C1': 1.525070127,                     │
│  'Shap_PaymentCard_Dummy1': 0.889305061, 'Shap_GNN_Emb_9': 0.445932445, 'Shap_Count_C5': 0.375499804,           │
│  'Shap_GNN_Emb_14': 0.33927135}.                                                                                │
│                                                                                                                 │
│          Provide a concise summary explaining to a human analyst why this transaction was flagged.              │
│                                                                                                                 │
│  ID: 3d355e98-c2ca-4572-9211-d529c85ce621                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: SHAP Interpreter Analyst                                                                                │
│                                                                                                                 │
│  Task:                                                                                                          │
│          Analyze Transaction 3663602 with Fraud Probability: 0.903028677.                                       │
│          The top driving SHAP features and their values are: {'Shap_Count_C1': 1.525070127,                     │
│  'Shap_PaymentCard_Dummy1': 0.889305061, 'Shap_GNN_Emb_9': 0.445932445, 'Shap_Count_C5': 0.375499804,           │
│  'Shap_GNN_Emb_14': 0.33927135}.                                                                                │
│                                                                                                                 │
│          Provide a concise summary explaining to a human analyst why this transaction was flagged.              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Fraud Triage Specialist                                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  1. Recommendation: keep blocked                                                                                │
│  2. Fraud probability: 0.903028677                                                                              │
│  3. Justification: The transaction 3663602 has a high fraud probability of 0.903028677, supported by several    │
│  alarming features such as high frequencies in 'Shap_Count_C1' suggesting suspicious behavior, and the payment  │
│  card associated with past fraudulent activities as indicated by 'Shap_PaymentCard_Dummy1'. Additionally, the   │
│  anomalies flagged by 'Shap_Count_C5' and the complex relationships captured in 'Shap_GNN_Emb_9' and            │
│  'Shap_GNN_Emb_14' reinforce the likelihood of fraud. The historical pattern from the Vector DB shows a         │
│  matching category of TP (True Positive), which aligns with the model's risk assessment and suggests that the   │
│  transaction should indeed be kept blocked to prevent potential fraud.                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│          Review the SHAP interpretation for Transaction 3663602 (Probability: 0.903028677).                     │
│          Here is the historical pattern matching result from the FAISS Vector Database search using Euclidean   │
│  distance on ALL SHAP vectors:                                                                                  │
│          Top 5 ANN Matching Categories from Vector DB (Based on ALL SHAP features): ['FP', 'TP', 'TP', 'FP',    │
│  'TP']                                                                                                          │
│  Euclidean Distances: [1.5302762985229492, 1.8340299129486084, 1.8703731298446655, 1.9167921543121338,          │
│  1.9924371242523193]                                                                                            │
│                                                                                                                 │
│  --> ANN Selected Category: TP                                                                                  │
│                                                                                                                 │
│          Combine the SHAP context with the Vector DB match to decide the best action.                           │
│          If the likely category is FP (False Positive), lean towards recommendations like "Request 3D           │
│  authentication" or "Allow with monitoring".                                                                    │
│          If TP (True Positive), lean towards "Keep blocked".                                                    │
│          If FN (False Negative), lean towards "Manual Review. Even though the probability is low, it might be   │
│  a fraud transaction".                                                                                          │
│                                                                                                                 │
│  Agent: Fraud Triage Specialist                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

1. Recommendation: keep blocked
2. Fraud probability: 0.903028677
3. Justification: The transaction 3663602 has a high fraud probability of 0.903028677, supported by several alarming features such as high frequencies in 'Shap_Count_C1' suggesting suspicious behavior, and the payment card associated with past fraudulent activities as indicated by 'Shap_PaymentCard_Dummy1'. Additionally, the anomalies flagged by 'Shap_Count_C5' and the complex relationships captured in 'Shap_GNN_Emb_9' and 'Shap_GNN_Emb_14' reinforce the likelihood of fraud. The historical pattern from the Vector DB shows a matching category of TP (True Positive), which aligns with the model's risk assessment and suggests that the transaction should indeed be kept blocked to prevent potential fraud.


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 1d7764a8-4d45-46cd-bb76-4912c612f8cd                                                                       │
│  Final Output: 1. Recommendation: keep blocked                                                                  │
│  2. Fraud probability: 0.903028677                                                                              │
│  3. Justification: The transaction 3663602 has a high fraud probability of 0.903028677, supported by several    │
│  alarming features such as high frequencies in 'Shap_Count_C1' suggesting suspicious behavior, and the payment  │
│  card associated with past fraudulent activities as indicated by 'Shap_PaymentCard_Dummy1'. Additionally, the   │
│  anomalies flagged by 'Shap_Count_C5' and the complex relationships captured in 'Shap_GNN_Emb_9' and            │
│  'Shap_GNN_Emb_14' reinforce the likelihood of fraud. The historical pattern from the Vector DB shows a         │
│  matching category of TP (True Positive), which aligns with the model's risk assessment and suggests that the   │
│  transaction should indeed be kept blocked to prevent potential fraud.                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [13]:
!streamlit run app.py

Usage: streamlit run [OPTIONS] [TARGET] [ARGS]...
Try 'streamlit run --help' for help.

Error: Invalid value: File does not exist: app.py


In [1]:
%%writefile fraud_agent.py
import os
import faiss
import numpy as np
import pandas as pd
from crewai import Agent, Task, Crew, Process
from dotenv import load_dotenv

# Load Google Drive colab environment variables for the API key
env_path = '/content/drive/MyDrive/Colab Notebooks/IEEE_CIS_Dataset/.env'
load_dotenv(env_path)

def build_vector_db(df_validation):
    shap_cols = [c for c in df_validation.columns if str(c).startswith('Shap_')]
    shap_cols.sort()
    vectors = np.ascontiguousarray(df_validation[shap_cols].fillna(0).values.astype('float32'))
    dim = vectors.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(vectors)
    categories = df_validation['Category'].values if 'Category' in df_validation.columns else np.array(['Unknown'] * len(df_validation))
    return index, shap_cols, categories

async def run_crewai_fraud_analysis(transaction_id, df_shap, vector_db):
    index, shap_cols, categories = vector_db
    tx_data = df_shap[df_shap['TransactionID'] == transaction_id]
    if tx_data.empty: 
        return None, f"Transaction {transaction_id} not found."
    
    probability = tx_data['Predicted_Fraud_Probability'].values[0] if 'Predicted_Fraud_Probability' in tx_data.columns else "N/A"
    
    top_5_series = tx_data[[c for c in shap_cols if c in tx_data.columns]].iloc[0].sort_values(ascending=False).head(5)
    tx_features_top_5 = top_5_series.to_dict()
    
    tx_vector_df = tx_data.reindex(columns=shap_cols, fill_value=0)
    tx_vector = np.ascontiguousarray(tx_vector_df.values.astype('float32'))
    
    distances, indices = index.search(tx_vector, 5)
    closest_categories = categories[indices[0]]
    most_likely_category = pd.Series(closest_categories).mode()[0]
    reference_data_sample_str = f"Top 5 ANN Matching Categories from Vector DB (Based on ALL SHAP features): {closest_categories.tolist()}\nEuclidean Distances: {distances[0].tolist()}\n\n--> ANN Selected Category: {most_likely_category}"
    
    shap_interpreter = Agent(
        role='SHAP Interpreter Analyst',
        goal='Interpret SHAP values to explain the key factors driving high fraud probabilities.',
        backstory='You are an expert Data Scientist specializing in model explainability and fraud detection.',
        verbose=True,
        allow_delegation=False,
        llm="gpt-4o-mini"
    )

    fraud_triage_specialist = Agent(
        role='Fraud Triage Specialist',
        goal='Provide an actionable recommendation based on SHAP explanations and Vector DB nearest-neighbor search.',
        backstory='You are a Senior Fraud Analyst referencing explainability metrics with historical FAISS patterns to make decisions.',
        verbose=True,
        allow_delegation=False,
        llm="gpt-4o-mini"
    )
    
    interpret_task = Task(
        description=f'''
        Analyze Transaction {transaction_id} with Fraud Probability: {probability}.
        Top features: {tx_features_top_5}.
        Summarize to a human analyst why this transaction was flagged.
        ''',
        expected_output='Brief paragraph explaining main reasons.',
        agent=shap_interpreter
    )

    triage_task = Task(
        description=f'''
        Review SHAP interpretation for Transaction {transaction_id} (Probability: {probability}).
        Historical pattern match from FAISS Vector DB: 
        {reference_data_sample_str}

        If FP (False Positive), recommend "Request 3D authentication" or "Allow". 
        If TP (True Positive), recommend "Keep blocked".
        If FN (False Negative), recommend "Manual Review".
        ''',
        expected_output=f'''
        STRICT format:
        1. Recommendation: [...]
        2. Fraud probability: [{probability}]
        3. Justification: [...]
        ''',
        agent=fraud_triage_specialist
    )

    crew = Crew(agents=[shap_interpreter, fraud_triage_specialist], tasks=[interpret_task, triage_task], verbose=True, process=Process.sequential)
    result = await crew.kickoff_async()
    return top_5_series, result

Writing fraud_agent.py


In [2]:
%%writefile app.py
import streamlit as st
import pandas as pd
import asyncio
import matplotlib.pyplot as plt
from fraud_agent import build_vector_db, run_crewai_fraud_analysis

# Page configuration
st.set_page_config(page_title="Fraud Triage Agentic AI", layout="wide", page_icon="🛡️")

@st.cache_resource
def load_data():
    # Use ABSOLUTE paths for Colab Google Drive
    df_shap_path = '/content/drive/MyDrive/Colab Notebooks/IEEE_CIS_Dataset/Output/Test_Feature_SHAP_Threshold_0.86.csv'
    df_val_path = '/content/drive/MyDrive/Colab Notebooks/IEEE_CIS_Dataset/Output/df_reference_sampled.csv'
    
    df_shap = pd.read_csv(df_shap_path)
    df_validation_refrence = pd.read_csv(df_val_path)
    
    vector_db = build_vector_db(df_validation_refrence)
    return df_shap, df_validation_refrence, vector_db

st.title("🛡️ Agentic AI - Fraud Triage Application")
st.markdown("This Streamlit app interacts with the underlying **CrewAI** agents and **FAISS Vector DB** to provide Explainable AI summaries for high-risk credit card transactions.")

try:
    df_shap, df_validation_refrence, vector_db = load_data()
except Exception as e:
    st.error(f"Error loading CSV data: {e}")
    st.stop()

st.markdown("---")
st.header("1. Analyst Input Query")

col1, col2 = st.columns([1, 2])
with col1:
    default_tx = df_shap['TransactionID'].iloc[0] if 'TransactionID' in df_shap.columns else "TEST_ID_1"
    transaction_id_input = st.text_input("Transaction ID", value=str(default_tx))
    try:
        transaction_id = int(transaction_id_input)
    except:
        transaction_id = transaction_id_input

with col2:
    question = st.text_input("Analyst Question", value="Why is the fraud probability so high?")

analyze_button = st.button("Run AI Fraud Triage Analysis", type="primary")

if analyze_button:
    with st.spinner("Agents are analyzing SHAP values & searching Historical Vector DB... Please wait."):
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        top_5_series, final_report = loop.run_until_complete(
            run_crewai_fraud_analysis(transaction_id, df_shap, vector_db)
        )
    
    if top_5_series is None:
        st.error(final_report)
    else:
        st.markdown("---")
        st.header("2. Fraud Triage Agent Output")
        st.success("Analysis complete.")
        st.markdown(f"**Analyst Query:** {question}")
        st.markdown(str(final_report))
        
        st.markdown("---")
        st.header("3. Top 5 SHAP Value Contributors Visualization")
        fig, ax = plt.subplots(figsize=(10, 5))
        top_5_series.sort_values(ascending=True).plot(kind='barh', color='salmon', ax=ax)
        ax.set_title(f'Top 5 SHAP Risk Features for Transaction {transaction_id}')
        ax.set_xlabel('SHAP Contribution Value')
        ax.set_ylabel('Variable Feature')
        plt.tight_layout()
        st.pyplot(fig)

Writing app.py


In [7]:
!pip install pyngrok

In [7]:


from pyngrok import ngrok
import time

# --- ACTION REQUIRED ---
# 1. Go to https://dashboard.ngrok.com/signup and create a free account
# 2. Go to https://dashboard.ngrok.com/get-started/your-authtoken
# 3. Copy your token and paste it inside the quotes below:
NGROK_AUTH_TOKEN = "HZWRz2h_7qJrLxW"

# Authenticate ngrok
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

print("Starting Streamlit app on background port 8501...")
# Kill any existing instances of Streamlit or ngrok occupying the port
!pkill -f streamlit
ngrok.kill()

# Run Streamlit silently in the background
get_ipython().system_raw('streamlit run app.py --server.enableCORS false --server.enableXsrfProtection false --server.headless true &')

time.sleep(3)

# Establish ngrok tunnel
public_url = ngrok.connect(8501)
print(f"===========================================================")
print(f"🚀 Streamlit App is LIVE at: {public_url}")
print(f"===========================================================")

Starting Streamlit app on background port 8501...


ERROR:pyngrok.process.ngrok:t=2026-08-07T01:00:26+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.\nYour authtoken: HZWRz2h_7qJrLxW\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n"
ERROR:pyngrok.process.ngrok:t=2026-08-07T01:00:26+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.\nYour authtoken: HZWRz2h_7qJrLxW\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n"
CRITICAL:pyngrok.process.ngrok:t=2026-08-07T01:00:26+0000 lvl=crit msg="command failed" err="authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.\nYour authtoken: 

PyngrokNgrokError: The ngrok process errored on start: authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.\nYour authtoken: HZWRz2h_7qJrLxW\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n.